# Tronco · un turno del relevo (Kaggle 2×T4)

Un turno: toma el candado en Hugging Face, sigue desde el checkpoint más nuevo (de Hugging Face o de uno de los dos respaldos de Kaggle), entrena ~11 h, sube un checkpoint cada 3 h mientras entrena, deja el último en el respaldo que toca, aplasta el historial y suelta el candado (`ludwig_lm/relay.py`, D15, P26).

**Desde el 25 de noviembre de 2026, el tronco es `tronco2`** (`configs/tronco_v2.json`, P30): empieza de cero con el router corregido, porque el balanceo de los expertos del primer intento se rompió en el paso ~1250 (P29). Los checkpoints del primer intento (`tronco`) quedan en Hugging Face bajo su etiqueta: cada corrida rota solo los suyos.

**Una vez por cuenta:**
1. Cuenta de Hugging Face, invitada a la organización del proyecto con rol *write*. Un token *fine-grained* con escritura solo en el repo del relevo.
2. En Kaggle, *Add-ons → Secrets*: `HF_TOKEN` (ese token), y `KAGGLE_USERNAME` y `KAGGLE_KEY` (el token de la API de tu cuenta, *Settings → API*). Cada uno usa el suyo.
3. Pipe te agrega como colaborador (*can edit*) en los dos datasets de respaldo y como lector en los tres de shards.

**Cada turno en un notebook nuevo,** importado del repo, con sus entradas recién agregadas. Cada respaldo crea una versión nueva de su dataset y borra las viejas (`kaggle datasets version -d`, para no llenar el espacio de Kaggle con checkpoints de 11 GB). Un notebook que ya tenía adjunta una versión vieja la ve como *no longer available*, y sin la última versión de los dos respaldos el turno no sabe cuál toca.

**En cada turno:** acelerador *GPU T4 x2*, Internet *On*, y como entradas todos los datasets de shards (`ludwig2-shards-1`, `-2`, …) y los dos de respaldo. Nunca los del build viejo (`ludwig2-tronco-v3-*`): el notebook elige el manifiesto con más shards, y podría quedarse con el viejo. Pon tu `CUENTA` ("cuenta 4", nunca el usuario) y *Save Version → Save & Run All*. Avisa por WhatsApp al empezar y al terminar.

| `MODO` | Qué hace | Tarda |
|---|---|---|
| `tronco` | Un turno del tronco | ~11.5 h |
| `prueba-relevo` | **G0-0:** el relevo completo con un modelo diminuto, contra `HF_REPO_PRUEBA`: candado, subida, aplastado, respaldo y reanudación exacta. Solo antes del primer turno del tronco | ~15 min |
| `prueba-respaldo` | Sigue desde el respaldo que dejó la última `prueba-relevo`, sin Hugging Face (como `SIN_CANDADO`): lo lee donde Kaggle lo montó y respalda en el otro dataset. Solo antes del primer turno del tronco | ~5 min |
| `g0-16k` | **G0-1:** 50 pasos del modelo real a 16K de contexto, sin guardar (condición 5 de la G0) | ~30–60 min |
| `sonda` | Mide el router de un checkpoint guardado en Hugging Face, por capa (P29), sin entrenar ni escribir nada | ~10 min |

In [ ]:
MODO = "tronco"          # tronco · sonda · prueba-relevo · prueba-respaldo · g0-16k
CUENTA = "cuenta 1"      # tu etiqueta en el relevo: nunca el usuario de Kaggle ni el correo
COMMIT = "048efa5d0a79830b40fa58068b68d439a3611d11"  # el commit del tronco2: el MISMO en todos sus turnos
REPO_URL = "https://github.com/FelipeGarciaRpo/ludwig-lm"
HF_REPO = "ludwig-v2-lm/ludwig-v2-lm-1"                       # privado, de la organización
HF_REPO_PRUEBA = "ludwig-v2-lm/ludwig-v2-lm-1-relevo-prueba"  # privado: solo para prueba-relevo
RESPALDO = ["DUENO/ludwig2-respaldo-a", "DUENO/ludwig2-respaldo-b"]   # los dos datasets privados de Pipe
SIN_CANDADO = False      # True solo si Hugging Face está caído y se coordinó por WhatsApp: sigue desde Kaggle
SONDA = ("tronco", 3270)  # MODO sonda: la etiqueta y el paso del checkpoint que se mide
import time
T_SESION = time.time()

In [ ]:
import os, subprocess, sys, json, glob, shutil
os.chdir("/kaggle/working")
if not os.path.exists("ludwig-lm"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "ludwig-lm"], check=True)
subprocess.run(["git", "-C", "ludwig-lm", "checkout", "-q", COMMIT], check=True)
os.chdir("/kaggle/working/ludwig-lm")
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
!pip install -q "transformers==5.22.2" safetensors sentencepiece protobuf hf_xet kaggle
from kaggle_secrets import UserSecretsClient
for k in ("HF_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[k] = UserSecretsClient().get_secret(k)
    except Exception:
        print(f"⚠️ falta el secreto {k}")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.insert(0, ".")
from ludwig_lm.relay import HFStore, KaggleBackup, Lock, LocalStore, Session, train_command, Busy
TRONCO = json.load(open("configs/tronco_v2.json"))       # P30: el tronco desde cero, con el router corregido
manifiestos = sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True),
                     key=lambda m: len(json.load(open(m))["shards"]))
MANIFIESTO = manifiestos[-1] if manifiestos else None
print("shards:", f"{len(json.load(open(MANIFIESTO))['shards'])} en {MANIFIESTO.split('/input/')[-1]}" if MANIFIESTO
      else "⚠️ ninguno: agrega los datasets de shards")
libre = shutil.disk_usage("/tmp").free / 2**30
print(f"/tmp: {libre:.0f} GiB libres" + ("" if libre >= 30 else
      " ⚠️ un turno necesita ~23 GiB (dos checkpoints de 11.3 GiB a la vez): avisa antes de entrenar"))
for slug in RESPALDO:                    # lo que Kaggle montó de cada respaldo, también los ocultos (.metadata)
    try:
        nombre, vistos, hay = slug.split("/")[-1], [], False
        for d, _, fs in os.walk("/kaggle/input"):
            partes = d.split("/")
            if nombre in partes:
                hay, dentro = True, "/".join(partes[partes.index(nombre) + 1:])
                vistos += [(os.path.join(dentro, f), os.path.getsize(os.path.join(d, f))) for f in fs]
        print(f"respaldo {nombre}:", "no está adjunto" if not hay else ", ".join(
            f"{p} ({s / 2**20:.0f} MiB)" for p, s in sorted(vistos)[:8]) or "vacío")
    except Exception as e:  # solo informa: nunca detiene el turno
        print(f"respaldo {slug.split('/')[-1]}: no se pudo listar ({type(e).__name__})")
import torch
if not torch.cuda.is_available():         # en CPU, train.py corre ~100 veces más lento y el turno se pierde
    raise SystemExit("⛔ sin GPU: en Settings → Accelerator elige GPU T4 x2 y vuelve a correr")
if MODO in ("tronco", "prueba-relevo", "prueba-respaldo") and any(s.split("/")[0] == "DUENO" for s in RESPALDO):
    raise SystemExit("⛔ RESPALDO todavía dice DUENO: pon el usuario de Kaggle de Pipe en los dos datasets")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## G0-0 · `prueba-relevo`

El relevo completo con el modelo diminuto (vocabulario de 64K, unos segundos por paso), contra `HF_REPO_PRUEBA` y los respaldos reales. Cada corrida usa su propia etiqueta (`prueba-relevo-<fecha y hora>`): el tronco la ignora, y la prueba se puede repetir sin limpiar lo que dejó la anterior.

**Solo antes del primer turno del tronco.** Cada versión de un dataset de respaldo guarda un solo checkpoint, así que una vez que empiece el tronco, esta prueba reemplazaría su respaldo.

1. La sesión A empieza de cero y entrena 30 pasos: guarda en el 10 (lo sube mientras entrena) y se detiene en el 20 como si se acabara el tiempo. Sube, respalda, aplasta y suelta.
2. Se prueba el candado: una sesión lo toma, otra recibe *ocupado*, y lo suelta.
3. La sesión B sigue desde Hugging Face hasta el paso 30. Su reanudación debe ser exacta en el estado y en el primer lote.

Al final imprime cada condición con ✅ o ❌. Lo que Kaggle montó de cada respaldo sale en la celda de preparación.

In [ ]:
if MODO == "prueba-relevo":
    ETIQUETA = time.strftime("prueba-relevo-%Y%m%d-%H%M%S", time.gmtime())   # nueva en cada corrida
    print("etiqueta:", ETIQUETA)
    c = json.load(open("configs/tiny_afmoe.json")); c.update(vocab_size=65536, pad_token_id=1, bos_token_id=0)
    json.dump(c, open("/tmp/tiny_64k.json", "w"))
    LOG = "/kaggle/working/results/relevo-prueba.jsonl"; os.makedirs(os.path.dirname(LOG), exist_ok=True)
    base = dict(config="/tmp/tiny_64k.json", tokenizer="tokenizer/tokenizer.json", shards=MANIFIESTO,
                shard_roots="/kaggle/input", seq=256, micro=4, tokens_per_step=2048, steps=30, warmup=5,
                eval_every=10, eval_windows=2, log_every=5, save_final=True, tag=ETIQUETA, out=LOG)
    tienda = HFStore(HF_REPO_PRUEBA, os.environ.get("HF_TOKEN"))
    respaldo = KaggleBackup(RESPALDO, tag=ETIQUETA)
    def sesion(extra):
        a = dict(base, **extra)
        # poll_s 0.5: el modelo diminuto va del paso 10 al 20 en ~3 s, y cada 10 s se perdía la subida del 10
        return Session(tienda, respaldo, CUENTA, ETIQUETA, f"/tmp/relevo-{len(extra)}",
                       lambda r, d: train_command(a, r, d), LOG, lease_s=1800, renew_s=600, poll_s=0.5).run()
    A = sesion({"save_at": [10], "pause_at_step": 20})
    l1, l2 = Lock(tienda, "prueba 1", "s1"), Lock(tienda, "prueba 2", "s2")
    h = l1.acquire()
    try:
        l2.acquire(); ocupado = False
    except Busy:
        ocupado = True
    l1.release(h)
    B = sesion({})
    filas = [json.loads(x) for x in open(LOG)]
    res = [r for r in filas if r["type"] == "resume"][-1:]
    lote = [r for r in filas if r["type"] == "resume_check"][-1:]
    subidas = {e["step"] for e in A["events"] + B["events"] if e["kind"] == "upload"}
    for e in A["events"] + B["events"]:
        print(" ", e["kind"], {k: v for k, v in e.items() if k not in ("kind", "time", "by", "session")})
    pruebas = {
        "A empezó de cero": A["resumed_from"]["where"] is None,
        "subió los pasos 10 (mientras entrenaba), 20 y 30": {10, 20, 30} <= subidas,
        "A dejó su checkpoint en un respaldo de Kaggle": any(e["kind"] == "backup" for e in A["events"]),
        "con el candado tomado, otra sesión recibe «ocupado»": ocupado,
        "B siguió desde Hugging Face en el paso 20": B["resumed_from"] == {"where": "hf", "step": 20},
        "reanudación exacta: el estado": bool(res and res[0]["state_exact"]),
        "reanudación exacta: el primer lote": bool(lote and lote[0]["batch_exact"]),
        "A y B aplastaron el historial y soltaron el candado":
            all(k in [e["kind"] for e in S["events"]] for S in (A, B) for k in ("squash", "release")),
    }
    print()
    for k, v in pruebas.items():
        print(" ", "✅" if v else "❌", k)
    ok = all(pruebas.values())
    print("\nG0-0:", "✅ el relevo funciona: subida, candado, respaldo, aplastado y reanudación exacta" if ok
          else "❌ revisar las condiciones y los eventos de arriba")

## Prueba del respaldo · `prueba-respaldo`

Un turno que sigue desde Kaggle, como con `SIN_CANDADO = True`: sin Hugging Face y sin candado. Toma el respaldo más nuevo de una `prueba-relevo`, lo lee donde Kaggle lo desempacó (sin copiarlo), entrena 10 pasos con el modelo diminuto y deja su checkpoint en el otro dataset de respaldo.

**Solo antes del primer turno del tronco,** por lo mismo que la G0-0: escribe en los respaldos reales, y cada versión guarda un solo checkpoint.

**En un notebook nuevo,** no en el de la `prueba-relevo`: esa prueba crea versiones nuevas de los respaldos y borra las que el notebook tenía adjuntas.

**Entradas:** los dos respaldos y los shards con los que se entrenó ese respaldo. La celda imprime cuáles son: el lector se niega a seguir con shards de otro build.

Pasa si sigue desde Kaggle en el paso del respaldo, sin copiarlo, con el estado y el primer lote exactos, y si respalda en el otro dataset.

In [ ]:
if MODO == "prueba-respaldo":
    vistos = []
    for slug in RESPALDO:
        nombre = slug.split("/")[-1]
        for m in glob.glob(f"/kaggle/input/**/{nombre}/checkpoint.json", recursive=True):
            meta = json.load(open(m))
            if str(meta.get("tag", "")).startswith("prueba-relevo"):
                vistos.append((meta["step"], meta.get("saved_at", 0), slug, meta))
    if not vistos:
        raise SystemExit("⛔ ningún respaldo de una prueba-relevo montado: adjunta ludwig2-respaldo-a y ludwig2-respaldo-b")
    _, _, slug, meta = max(vistos, key=lambda v: v[:2])
    print(f"respaldo: {slug.split('/')[-1]}, paso {meta['step']}, etiqueta {meta['tag']}")
    print(f"  se entrenó con {meta['args']['shards']}")
    print(f"  shards adjuntos: {MANIFIESTO}")
    c = json.load(open("configs/tiny_afmoe.json")); c.update(vocab_size=65536, pad_token_id=1, bos_token_id=0)
    json.dump(c, open("/tmp/tiny_64k.json", "w"))
    LOG = "/kaggle/working/results/respaldo-prueba.jsonl"; os.makedirs(os.path.dirname(LOG), exist_ok=True)
    a = dict(config="/tmp/tiny_64k.json", tokenizer="tokenizer/tokenizer.json", shards=MANIFIESTO,
             shard_roots="/kaggle/input", seq=256, micro=4, tokens_per_step=2048, steps=meta["step"] + 10, warmup=5,
             eval_every=10, eval_windows=2, log_every=5, save_final=True, tag=meta["tag"], out=LOG)
    S = Session(None, KaggleBackup(RESPALDO, tag=meta["tag"]), CUENTA, meta["tag"], "/tmp/respaldo-prueba",
                lambda r, d: train_command(a, r, d), LOG, no_lock=True, poll_s=0.5).run()
    filas = [json.loads(x) for x in open(LOG)] if os.path.exists(LOG) else []
    res = [r for r in filas if r["type"] == "resume"][-1:]
    lote = [r for r in filas if r["type"] == "resume_check"][-1:]
    for e in S["events"]:
        print(" ", e["kind"], {k: v for k, v in e.items() if k not in ("kind", "time", "by", "session")})
    pruebas = {
        f"siguió desde Kaggle en el paso {meta['step']}": S["resumed_from"] == {"where": "kaggle", "step": meta["step"]},
        "leyó el respaldo donde Kaggle lo montó, sin copiarlo": any(e["kind"] == "in_place" for e in S["events"]),
        "reanudación exacta: el estado": bool(res and res[0]["state_exact"]),
        "reanudación exacta: el primer lote": bool(lote and lote[0]["batch_exact"]),
        f"entrenó hasta el paso {meta['step'] + 10}": S["train_exit"] == 0 and S["last_local"] == f"step_{meta['step'] + 10:06d}",
        "respaldó en el otro dataset": any(e["kind"] == "backup" and e["dataset"] != slug for e in S["events"]),
    }
    print()
    for k, v in pruebas.items():
        print(" ", "✅" if v else "❌", k)
    print("\nRespaldo:", "✅ un turno puede seguir desde Kaggle" if all(pruebas.values())
          else "❌ revisar las condiciones y los eventos de arriba")

## G0-1 · `g0-16k`

50 pasos del modelo real a 16K de contexto (un lote de 16,384 tokens por GPU), con los shards reales, sin guardar. Pasa si termina sin falta de memoria.

In [ ]:
if MODO == "g0-16k":
    LOG = "/kaggle/working/results/g0-16k.jsonl"; os.makedirs(os.path.dirname(LOG), exist_ok=True)
    a = dict(TRONCO["args"], shards=MANIFIESTO, shard_roots="/kaggle/input", seq=16384, micro=1,
             tokens_per_step=32768, ramp_tokens=0, steps=50, warmup=10, eval_every=0, eval_windows=2,
             save_every_hours=0, save_final=False, keep=1, log_every=5, tag="g0-16k", out=LOG)
    p = subprocess.run(train_command(a, None, "/tmp/g0_16k"))
    fin = [json.loads(x) for x in open(LOG) if json.loads(x)["type"] == "end"][-1:] if os.path.exists(LOG) else []
    if p.returncode == 0 and fin and fin[0]["status"] == "done":
        f = fin[0]
        print(f"G0-1 ✅ 50 pasos a 16K sin falta de memoria: pico {f['peak_reserved_GiB']} GiB reservada, "
              f"{f['tokens_per_s_mean']} tokens/s")
    else:
        print("G0-1 ❌ no terminó: ver el log de arriba (¿out of memory?)")

## Sonda del router · `sonda`

Baja de Hugging Face el checkpoint de `SONDA` (etiqueta y paso) y lo evalúa con `train.py --eval-only`, en las dos GPUs: la validación y, por capa MoE, el estado del router (P29). No toma el candado ni escribe nada en Hugging Face ni en Kaggle, así que puede correr mientras otra cuenta tiene un turno.

Por capa imprime:
- **la brecha:** la mediana, sobre los tokens, de la diferencia entre el score del experto k-ésimo y el del (k+1)-ésimo;
- **λ/brecha:** cuánto puede mover SMEBU esa elección en un paso. Con más de ~5, un paso del sesgo re-enruta a casi todos los tokens a la vez (en el modelo de juguete del P29: estable hasta ~4, colapso desde ~10);
- **el rango del sesgo** y la fracción de tokens cuyos expertos cambian por el sesgo;
- **el experto más cargado** en la validación, en veces la media.

En el checkpoint 3270 del primer intento, el diagnóstico del P29 queda confirmado si λ/brecha está alto en las primeras 5 capas MoE y bajo en las otras 20. **Entradas:** los datasets de shards, como en un turno.

In [ ]:
if MODO == "sonda":
    etiqueta, paso = SONDA
    tienda = HFStore(HF_REPO, os.environ.get("HF_TOKEN"))          # solo lee: ni candado ni commits
    cabeza = tienda.head()
    idx = json.loads(tienda.read("checkpoints/index.json", cabeza))["checkpoints"]
    elegido = [c for c in idx if c.get("tag") == etiqueta and c["step"] == paso]
    if not elegido:
        raise SystemExit(f"⛔ {HF_REPO} no tiene {etiqueta} {paso}; tiene: "
                         + ", ".join(f"{c.get('tag')} {c['step']}" for c in idx))
    t = time.time()
    ruta = tienda.download(elegido[0]["path"], cabeza, "/tmp/sonda")
    print(f"checkpoint {etiqueta} {paso}: bajado en {time.time() - t:.0f} s")
    base = json.load(open("configs/tronco_v1.json")) if etiqueta == "tronco" else TRONCO
    LOG = f"/kaggle/working/results/tronco/sonda-{etiqueta}-{paso}.jsonl"
    os.makedirs(os.path.dirname(LOG), exist_ok=True)
    a = dict(base["args"], shards=MANIFIESTO, shard_roots="/kaggle/input", tag=f"sonda-{etiqueta}", out=LOG,
             eval_only=True)
    p = subprocess.run(train_command(a, str(ruta), "/tmp/sonda_ckpt"))
    ev = [r for r in map(json.loads, open(LOG)) if r["type"] == "eval"][-1:] if os.path.exists(LOG) else []
    if p.returncode or not ev:
        raise SystemExit("⛔ la sonda no terminó: ver el log de arriba")
    e = ev[0]
    print(f"\nval bpb {e['val_bpb']} · λ de SMEBU en ese paso {e['lam']}")
    print("capa MoE |  brecha | λ/brecha | rango del sesgo | el sesgo decide | más cargado")
    for i, (m, d, b, c) in enumerate(zip(e["router_margin"], e["bias_decides"], e["bias_range"],
                                         e["val_load_max_over_mean"])):
        print(f"{i:8d} | {m:7.1e} | {e['lam'] / max(m, 1e-12):8.1f} | {b:15.3f} | {d:15.0%} | {c:9.2f}×")

## Tronco · un turno

Entrena hasta ~11 h desde que empezó la sesión y deja ~45 min para la subida final, el respaldo (~11 GB a Kaggle) y el aplastado. Cada línea `[relevo]` es un evento: candado, de dónde siguió, cada subida, lo que se borra del disco, el respaldo y la entrega.

En `/tmp` nunca hay más de dos checkpoints a la vez (~23 GiB). El de partida se borra después del primer guardado nuevo, se guarda localmente solo el periódico más nuevo, y antes del respaldo solo queda el último.

**Qué mirar mientras corre** (P30). Si pasa alguna de estas cosas, avisa por WhatsApp; el turno sigue solo:
- **`z_router`** en cada línea `[train]`: el nivel del router. Debería quedarse alto. Si baja de forma sostenida, avisa; si sale `[alarm] … z_router_below_alarm`, quedó por debajo de 1.5 durante 50 pasos, que es lo que anticipó el quiebre del primer tronco.
- **`[eval] watch: MoE layer …`**, en las evaluaciones (cada 250 pasos): una capa se acerca a las rotas del primer tronco (el sesgo decide más del 80% de los tokens, su rango pasa de 20× la brecha, o el experto más cargado pasa de 2×).
- **`load max/mean`** debería rondar 1.5. Si el balanceo se rompe, el turno se detiene solo y guarda (`stopped_balance`): no lo relances.

In [ ]:
if MODO == "tronco":
    LOG = f"/kaggle/working/results/tronco/{time.strftime('%Y%m%d-%H%M')}-{CUENTA.replace(' ', '-')}.jsonl"
    os.makedirs(os.path.dirname(LOG), exist_ok=True)
    horas = round(11.0 - (time.time() - T_SESION) / 3600, 2)
    a = dict(TRONCO["args"], shards=MANIFIESTO, shard_roots="/kaggle/input", max_hours=horas, tag=TRONCO["tag"], out=LOG)
    tienda = None if SIN_CANDADO else HFStore(HF_REPO, os.environ.get("HF_TOKEN"))
    if not SIN_CANDADO:
        quien = Lock(tienda, CUENTA, "-").holder(tienda.head())
        if quien and not quien.get("released") and quien["expires"] > time.time():
            print(f"candado: lo tiene {quien['holder']} por {(quien['expires'] - time.time()) / 60:.0f} min más")
        else:
            print("candado: libre" + (f" (el último turno fue de {quien['holder']})" if quien else ""))
    s = Session(tienda, KaggleBackup(RESPALDO, tag=TRONCO["tag"]), CUENTA, TRONCO["tag"], "/tmp/relevo",
                lambda r, d: train_command(a, r, d), LOG, no_lock=SIN_CANDADO)
    if SIN_CANDADO and s.newest()[0] is None:   # sin Hugging Face, solo un respaldo dice dónde va el tronco
        raise SystemExit("⛔ SIN_CANDADO y ningún respaldo del tronco montado: no se entrena desde cero. "
                         "Adjunta ludwig2-respaldo-a y ludwig2-respaldo-b.")
    try:
        RESUMEN = s.run()
    except Busy as e:
        raise SystemExit(f"⛔ el relevo está ocupado: {e}. No entrenes: avisa por WhatsApp.")
    json.dump(RESUMEN, open(LOG.replace(".jsonl", ".relevo.json"), "w"), indent=1)
    if not SIN_CANDADO and any(e["kind"] == "release_failed" for e in RESUMEN["events"]):
        # Turno 1: la entrega falló justo después del aplastado (Hugging Face aún mostraba la cabeza vieja).
        # Se reintenta con la cabeza nueva, y solo si el candado sigue siendo de esta sesión.
        candado = Lock(tienda, CUENTA, RESUMEN["session"])
        for intento in range(1, 7):
            time.sleep(30)
            try:
                cabeza = tienda.head()
                if (candado.holder(cabeza) or {}).get("session") != RESUMEN["session"]:
                    print("candado: ya no es de este turno, no se toca")
                    break
                candado.release(cabeza)
                print(f"candado: soltado al reintentar (intento {intento})")
                break
            except Exception as e:  # noqa: BLE001 - si no se puede, vence solo a las 2 h de su última renovación
                print(f"candado: el intento {intento} falló ({type(e).__name__}: {str(e)[:120]})")
    filas = [json.loads(x) for x in open(LOG)]
    fin = ([r for r in filas if r["type"] == "end"] or [{}])[-1]
    res = [r for r in filas if r["type"] == "resume"][-1:]
    lote = [r for r in filas if r["type"] == "resume_check"][-1:]
    subidas = [e for e in RESUMEN["events"] if e["kind"] == "upload"]
    tps, pico = fin.get("tokens_per_s_last_hour"), fin.get("peak_reserved_GiB")
    print(f"\n{fin.get('status')}: paso {fin.get('steps_done')}, {fin.get('tokens', 0) / 1e9:.3f}B tokens, "
          f"{fin.get('hours_this_session')} h en esta sesión")
    print("Condiciones de la G0 en este turno:")
    print(f"  1. ≥6 h sin NaN, escala estable: {'✅' if (fin.get('hours_this_session') or 0) >= 6 and fin.get('status') != 'error' and (fin.get('loss_scale_last') or 0) >= 1 else '❌'} "
          f"({fin.get('loss_scale_skips')} saltos de FP16, escala final {fin.get('loss_scale_last')})")
    print(f"  2. ≥2,600 tokens/s en la última hora: {'✅' if tps and tps >= 2600 else '❌'} ({tps})")
    print(f"  3. ≤13.5 GiB reservada: {'✅' if pico and pico <= 13.5 else '❌'} ({pico})")
    print(f"  4. ≥2 subidas: {'✅' if len(subidas) >= 2 else '❌'} ({len(subidas)}); reanudación en otra cuenta: "
          + (f"{'✅' if res[0]['state_exact'] and lote and lote[0]['batch_exact'] else '❌'} (desde {RESUMEN['resumed_from']})"
             if res else "— (este turno empezó de cero)"))

## Qué hacer con el resultado

1. **Del turno:**
   - De la salida (`results/tronco/`): el `.jsonl` del turno y su `.relevo.json`, con los eventos del relevo.
   - Van al repo en `results/tronco/`, con su fila en `results/README.md` y una entrada en la bitácora. El log también queda en `logs/` del repo de Hugging Face.
   - De una `sonda`: su `.jsonl` (`results/tronco/sonda-<etiqueta>-<paso>.jsonl`) y la tabla que imprimió, igual.
2. **WhatsApp:** "relevo libre, paso N". Si el resumen dice `lock_lost`, otra sesión tomó el candado después de que este venciera, y esta paró sola: avisa.
3. **La G0** está completa cuando:
   - el primer turno del tronco pasa las condiciones 1–4 (menos la reanudación);
   - el segundo turno, en otra cuenta, marca la reanudación ✅;
   - `g0-16k` dio ✅.

   Si algo falla, el checkpoint del primer turno sigue sirviendo salvo que lo que falló sea el entrenamiento mismo (NaN, escala).